# Step 1 — Scratch Mastery Environment and Path Setup

## Step 1.1 — Notebook Purpose

This notebook establishes the complete technical foundation for the Scratch Mastery pipeline.

It will:

- Define and validate all canonical project paths.
- Validate the official training feature and label files.
- Create an independent Scratch Mastery output workspace.
- Inventory the available raw dataset files.
- Discover or register the raw transcript source.
- Validate the response, session, objective, and target schema.
- Recover the frozen five-fold grouped validation assignments.
- Confirm that no tutoring session appears in multiple folds.
- Record Python, package, Git, CPU, memory, disk, and GPU information.
- Save reusable environment, path, fold, and setup manifests.

## Step 1.2 — Important Rules

- This notebook does not perform exploratory data analysis.
- This notebook does not parse transcript turns.
- This notebook does not train any machine-learning model.
- This notebook does not fit TF-IDF, embedding, or transformer models.
- The new pipeline will be built from the official raw data.
- The previous master dataset is an optional verification artifact only.
- Previous baseline model files are read-only references.
- The previous frozen grouped folds will be retained for fair baseline comparison.
- No package will be installed automatically inside this notebook.
- No existing baseline output will be overwritten.

# Step 2 — Core Imports and Global Settings

## Step 2.1 — Scope

This step imports only lightweight packages required for project setup, file validation, manifest creation, environment inspection, and table processing.

Heavy modelling libraries will not be loaded unless they are required for hardware detection.

In [1]:
from pathlib import Path
from datetime import datetime
from importlib.metadata import PackageNotFoundError, version
from IPython.display import display
import hashlib
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Setup time: {datetime.now().astimezone().isoformat()}")
print(f"Python executable: {sys.executable}")
print(f"Current working directory: {Path.cwd()}")

Setup time: 2026-08-07T02:04:01.491176+06:00
Python executable: c:\Users\USER\miniconda3\envs\ml_project\python.exe
Current working directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook_advance\00_project_setup


# Step 3 — Canonical Project Paths

## Step 3.1 — Path Policy

All project paths are defined in one place.

The notebook will not depend on the current working directory. Every later step will use the canonical project root defined below.

`TRANSCRIPT_SOURCE_OVERRIDE` should remain `None` during the first run. After transcript discovery, it can be replaced with the exact transcript file or folder path when manual selection is required.

In [2]:
PROJECT_ROOT = Path(r"C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition")
DATA_ROOT = PROJECT_ROOT / "Trace-The-Race-Dataset"
OLD_NOTEBOOK_ROOT = PROJECT_ROOT / "notebook"
MODEL_ANALYSIS_ROOT = OLD_NOTEBOOK_ROOT / "model_analysis"
BASELINE_REFERENCE_ROOT = MODEL_ANALYSIS_ROOT / "baselineanalysis"
ADVANCED_NOTEBOOK_ROOT = PROJECT_ROOT / "notebook_advance"
SCRATCH_OUTPUT_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"
SETUP_OUTPUT_DIR = SCRATCH_OUTPUT_ROOT / "00_project_setup"

TRAIN_LABELS_PATH = DATA_ROOT / "train_labels_44ujmj2.csv"
TRAIN_FEATURES_PATH = DATA_ROOT / "train_features_TMQTWsB.csv"
SUBMISSION_FORMAT_PATH_1 = DATA_ROOT / "submission_format_ZQLcKx7.csv"
SUBMISSION_FORMAT_PATH_2 = DATA_ROOT / "submission_format_5muR4s3.csv"
OLD_MASTER_DATASET_PATH = DATA_ROOT / "outputs" / "03_master_dataset" / "master_train.parquet"

BASELINE_FOLD_DESIGN_ROOT = BASELINE_REFERENCE_ROOT / "02_oof_analysis" / "00_fold_design"
BASELINE_FOLD_MODELS_ROOT = BASELINE_REFERENCE_ROOT / "02_oof_analysis" / "01_fold_models"
BASELINE_OOF_ROOT = BASELINE_REFERENCE_ROOT / "02_oof_analysis" / "02_oof_predictions"

BASELINE_MODEL_NOTEBOOK = MODEL_ANALYSIS_ROOT / "baseline_model_analysis.ipynb"
BASELINE_ANALYSIS_NOTEBOOK = MODEL_ANALYSIS_ROOT / "baseline_analysis_2.ipynb"
BASELINE_DIAGNOSTIC_NOTEBOOK = MODEL_ANALYSIS_ROOT / "Step_7_Integrated_Diagnostic_8_Cells (1).ipynb"
BASELINE_REPORT_ROOT = MODEL_ANALYSIS_ROOT / "Baseline_analysis_report"

TRANSCRIPT_SOURCE_OVERRIDE = None

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {DATA_ROOT}")
print(f"Baseline reference root: {BASELINE_REFERENCE_ROOT}")
print(f"Advanced notebook root: {ADVANCED_NOTEBOOK_ROOT}")
print(f"Scratch output root: {SCRATCH_OUTPUT_ROOT}")

Project root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
Dataset root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
Baseline reference root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis
Advanced notebook root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook_advance
Scratch output root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs


# Step 4 — Reusable Helper Functions

## Step 4.1 — Helper Responsibilities

The helper functions below will:

- Save dictionaries as JSON files.
- Load CSV, Parquet, JSON, and JSONL tables.
- Detect required columns without silently renaming unknown columns.
- Calculate SHA-256 file fingerprints.
- Run Git commands safely.
- Report file and directory availability.
- Save DataFrames as Parquet with a CSV fallback.
- Prepare tables whose identifiers may be stored in the index.

In [3]:
def write_json(data, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

def read_table(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    raise ValueError(f"Unsupported table format: {path}")

def prepare_table(dataframe):
    prepared = dataframe.copy()
    if not isinstance(prepared.index, pd.RangeIndex) or prepared.index.name is not None:
        prepared = prepared.reset_index()
    return prepared

def pick_column(dataframe, candidates, table_name, required=True):
    column_lookup = {str(column).strip().lower(): column for column in dataframe.columns}
    for candidate in candidates:
        if candidate.lower() in column_lookup:
            return column_lookup[candidate.lower()]
    if required:
        raise KeyError(f"None of the expected columns {candidates} were found in {table_name}. Available columns: {dataframe.columns.tolist()}")
    return None

def file_sha256(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()
    with path.open("rb") as file_object:
        while True:
            chunk = file_object.read(chunk_size)
            if not chunk:
                break
            sha256.update(chunk)
    return sha256.hexdigest()

def run_command(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, shell=False)
    return {"returncode": result.returncode, "stdout": result.stdout.strip(), "stderr": result.stderr.strip()}

def path_status(name, path, required):
    resource_type = "directory" if path.is_dir() else "file" if path.is_file() else "missing"
    return {"resource": name, "required": required, "exists": path.exists(), "type": resource_type, "path": str(path)}

def save_dataframe(dataframe, parquet_path):
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        dataframe.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as error:
        csv_path = parquet_path.with_suffix(".csv")
        dataframe.to_csv(csv_path, index=False)
        print(f"Parquet save failed: {error}")
        print(f"CSV fallback saved: {csv_path}")
        return csv_path

print("Helper functions are ready.")

Helper functions are ready.


# Step 5 — Project Validation and Workspace Creation

## Step 5.1 — Required Root Checks

This step confirms that:

- The project root exists.
- The project is a Git repository.
- The official dataset root exists.
- The old notebook root exists.
- The new advanced notebook root exists.

## Step 5.2 — Independent Workspace

The Scratch Mastery pipeline will use a separate output root so that no old baseline artifact is overwritten.

In [4]:
assert PROJECT_ROOT.exists(), f"Project root not found: {PROJECT_ROOT}"
assert (PROJECT_ROOT / ".git").exists(), f"Git repository not found: {PROJECT_ROOT}"
assert DATA_ROOT.exists(), f"Dataset root not found: {DATA_ROOT}"
assert OLD_NOTEBOOK_ROOT.exists(), f"Old notebook root not found: {OLD_NOTEBOOK_ROOT}"
assert ADVANCED_NOTEBOOK_ROOT.exists(), f"Advanced notebook root not found: {ADVANCED_NOTEBOOK_ROOT}"

OUTPUT_STAGE_NAMES = ["00_project_setup", "01_data_foundation", "02_advanced_eda", "03_retrieval", "04_evidence_pack", "05_reranker", "06_mastery_model", "07_oof_evaluation", "08_ensemble", "09_submission"]
NOTEBOOK_STAGE_NAMES = ["00_project_setup", "01_data_foundation", "02_advanced_eda", "03_retrieval", "04_feature_engineering", "05_model_training", "06_oof_evaluation", "07_ensemble", "08_submission"]

for stage_name in OUTPUT_STAGE_NAMES:
    (SCRATCH_OUTPUT_ROOT / stage_name).mkdir(parents=True, exist_ok=True)

for stage_name in NOTEBOOK_STAGE_NAMES:
    (ADVANCED_NOTEBOOK_ROOT / stage_name).mkdir(parents=True, exist_ok=True)

print(f"Canonical project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")
print(f"Scratch workspace ready: {SCRATCH_OUTPUT_ROOT}")
print(f"Advanced notebook workspace ready: {ADVANCED_NOTEBOOK_ROOT}")

Canonical project root: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
Current working directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook_advance\00_project_setup
Scratch workspace ready: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs
Advanced notebook workspace ready: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook_advance


# Step 6 — Source Registry and File Validation

## Step 6.1 — Required Sources

The following sources are required now:

- Official training features.
- Official training labels.
- Project Git repository.
- Baseline reference root for frozen-fold recovery.

## Step 6.2 — Optional Sources

The following sources are optional during setup:

- Previous master dataset.
- Submission templates.
- Previous OOF output folder.
- Final baseline notebooks.
- Baseline report folder.

In [5]:
source_registry_rows = [
    path_status("Project root", PROJECT_ROOT, True),
    path_status("Git repository", PROJECT_ROOT / ".git", True),
    path_status("Dataset root", DATA_ROOT, True),
    path_status("Training features", TRAIN_FEATURES_PATH, True),
    path_status("Training labels", TRAIN_LABELS_PATH, True),
    path_status("Baseline reference root", BASELINE_REFERENCE_ROOT, True),
    path_status("Baseline fold design root", BASELINE_FOLD_DESIGN_ROOT, False),
    path_status("Baseline fold models root", BASELINE_FOLD_MODELS_ROOT, False),
    path_status("Baseline OOF root", BASELINE_OOF_ROOT, False),
    path_status("Old master dataset", OLD_MASTER_DATASET_PATH, False),
    path_status("Submission format 1", SUBMISSION_FORMAT_PATH_1, False),
    path_status("Submission format 2", SUBMISSION_FORMAT_PATH_2, False),
    path_status("Baseline model notebook", BASELINE_MODEL_NOTEBOOK, False),
    path_status("Baseline analysis notebook", BASELINE_ANALYSIS_NOTEBOOK, False),
    path_status("Baseline diagnostic notebook", BASELINE_DIAGNOSTIC_NOTEBOOK, False),
    path_status("Baseline report folder", BASELINE_REPORT_ROOT, False),
    path_status("Advanced notebook root", ADVANCED_NOTEBOOK_ROOT, True),
    path_status("Scratch output root", SCRATCH_OUTPUT_ROOT, True)
]

source_registry = pd.DataFrame(source_registry_rows)
missing_required_sources = source_registry.loc[source_registry["required"] & ~source_registry["exists"]].copy()

display(source_registry)
assert missing_required_sources.empty, f"Required sources are missing:\n{missing_required_sources[['resource', 'path']].to_string(index=False)}"

print("All currently required project sources are available.")

,resource,required,exists,type,path
0,Project root,True,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
1,Git repository,True,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\.git
2,Dataset root,True,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
3,Training features,True,True,file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_features_TMQTWsB.csv
4,Training labels,True,True,file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_labels_44ujmj2.csv
5,Baseline reference root,True,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis
6,Baseline fold design root,False,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis\02_oof_analysis\00_fold_design
7,Baseline fold models root,False,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis\02_oof_analysis\01_fold_models
8,Baseline OOF root,False,True,directory,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis\02_oof_analysis\02_oof_predictions
9,Old master dataset,False,True,file,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\outputs\03_master_dataset\master_train.parquet


All currently required project sources are available.


# Step 6.3 — Official Source Fingerprints

SHA-256 fingerprints will be recorded for the official training feature and label files.

These fingerprints make it possible to confirm later that every experiment used the same original source data.

In [6]:
source_fingerprints = {
    "created_at": datetime.now().astimezone().isoformat(),
    "train_features_path": str(TRAIN_FEATURES_PATH),
    "train_features_size_bytes": TRAIN_FEATURES_PATH.stat().st_size,
    "train_features_sha256": file_sha256(TRAIN_FEATURES_PATH),
    "train_labels_path": str(TRAIN_LABELS_PATH),
    "train_labels_size_bytes": TRAIN_LABELS_PATH.stat().st_size,
    "train_labels_sha256": file_sha256(TRAIN_LABELS_PATH)
}

write_json(source_fingerprints, SETUP_OUTPUT_DIR / "source_fingerprints.json")
display(pd.DataFrame([source_fingerprints]).T.rename(columns={0: "value"}))

,value
created_at,2026-08-07T02:07:25.640058+06:00
train_features_path,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_features_TMQTWsB.csv
train_features_size_bytes,2370758
train_features_sha256,71bea3abb76a1cff5e1eaa75b9cbcfaf26d0419f6274b83a199ed520047a5063
train_labels_path,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_labels_44ujmj2.csv
train_labels_size_bytes,420887
train_labels_sha256,d98ee4389e5cde3f66d6d15b7b574261024a80e405958eca333d3c1921fd65b9


# Step 7 — Raw Dataset Inventory and Transcript Discovery

## Step 7.1 — Inventory Scope

The official dataset root will be scanned for raw source files.

Files inside the previous generated `outputs` directory will be excluded from transcript discovery.

## Step 7.2 — Transcript Keywords

Possible transcript sources will be identified using these path keywords:

- transcript
- utterance
- dialogue
- conversation
- message

A discovered candidate is not automatically accepted unless it is the only clear candidate or an exact override path has been provided.

In [7]:
SUPPORTED_RAW_EXTENSIONS = {".csv", ".json", ".jsonl", ".parquet", ".pq", ".txt", ".zip"}
TRANSCRIPT_KEYWORDS = ["transcript", "utterance", "dialogue", "conversation", "message"]

raw_inventory_rows = []

for file_path in DATA_ROOT.rglob("*"):
    if not file_path.is_file():
        continue
    relative_path = file_path.relative_to(DATA_ROOT)
    if "outputs" in {part.lower() for part in relative_path.parts}:
        continue
    if file_path.suffix.lower() not in SUPPORTED_RAW_EXTENSIONS:
        continue
    raw_inventory_rows.append({"name": file_path.name, "extension": file_path.suffix.lower(), "size_mb": round(file_path.stat().st_size / (1024 ** 2), 4), "parent": str(file_path.parent), "relative_path": str(relative_path), "absolute_path": str(file_path)})

raw_data_inventory = pd.DataFrame(raw_inventory_rows)

if not raw_data_inventory.empty:
    raw_data_inventory = raw_data_inventory.sort_values(["parent", "name"]).reset_index(drop=True)

raw_data_inventory.to_csv(SETUP_OUTPUT_DIR / "raw_data_inventory.csv", index=False)

print(f"Raw source files found: {len(raw_data_inventory):,}")
display(raw_data_inventory.head(50))

Raw source files found: 22,825


,name,extension,size_mb,parent,relative_path,absolute_path
0,submission_format_5muR4s3.csv,.csv,0.0012,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset,submission_format_5muR4s3.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\submission_format_5muR4s3.csv
1,submission_format_ZQLcKx7.csv,.csv,0.1203,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset,submission_format_ZQLcKx7.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\submission_format_ZQLcKx7.csv
2,train_features_TMQTWsB.csv,.csv,2.2609,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset,train_features_TMQTWsB.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_features_TMQTWsB.csv
3,train_labels_44ujmj2.csv,.csv,0.4014,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset,train_labels_44ujmj2.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_labels_44ujmj2.csv
4,aaaedit.csv,.csv,0.0225,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aaaedit.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aaaedit.csv
5,aaaptjd.csv,.csv,0.0409,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aaaptjd.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aaaptjd.csv
6,aabkeov.csv,.csv,0.0231,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aabkeov.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aabkeov.csv
7,aacggvb.csv,.csv,0.0269,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aacggvb.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aacggvb.csv
8,aadexbc.csv,.csv,0.0160,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aadexbc.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aadexbc.csv
9,aadinwu.csv,.csv,0.0179,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,train_transcripts\aadinwu.csv,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts\aadinwu.csv


In [8]:
if raw_data_inventory.empty:
    transcript_candidates = pd.DataFrame(columns=["name", "extension", "size_mb", "parent", "relative_path", "absolute_path"])
else:
    transcript_keyword_pattern = "|".join(TRANSCRIPT_KEYWORDS)
    transcript_candidate_mask = raw_data_inventory["relative_path"].str.lower().str.contains(transcript_keyword_pattern, regex=True)
    transcript_candidates = raw_data_inventory.loc[transcript_candidate_mask].copy()

if transcript_candidates.empty:
    transcript_candidate_summary = pd.DataFrame(columns=["parent", "file_count", "total_size_mb", "extensions"])
else:
    transcript_candidate_summary = transcript_candidates.groupby("parent").agg(file_count=("absolute_path", "size"), total_size_mb=("size_mb", "sum"), extensions=("extension", lambda values: ", ".join(sorted(set(values))))).reset_index()
    transcript_candidate_summary = transcript_candidate_summary.sort_values(["file_count", "total_size_mb"], ascending=False).reset_index(drop=True)

directory_candidates = []

for directory_path in DATA_ROOT.rglob("*"):
    if not directory_path.is_dir():
        continue
    relative_directory = directory_path.relative_to(DATA_ROOT)
    if "outputs" in {part.lower() for part in relative_directory.parts}:
        continue
    if any(keyword in str(relative_directory).lower() for keyword in TRANSCRIPT_KEYWORDS):
        directory_candidates.append(directory_path)

directory_candidates = sorted(set(directory_candidates), key=lambda path: str(path))

if TRANSCRIPT_SOURCE_OVERRIDE is not None:
    TRANSCRIPT_SOURCE = Path(TRANSCRIPT_SOURCE_OVERRIDE)
    assert TRANSCRIPT_SOURCE.exists(), f"Transcript source override does not exist: {TRANSCRIPT_SOURCE}"
elif len(directory_candidates) == 1:
    TRANSCRIPT_SOURCE = directory_candidates[0]
elif len(transcript_candidate_summary) == 1:
    TRANSCRIPT_SOURCE = Path(transcript_candidate_summary.iloc[0]["parent"])
elif len(transcript_candidates) == 1:
    TRANSCRIPT_SOURCE = Path(transcript_candidates.iloc[0]["absolute_path"])
else:
    TRANSCRIPT_SOURCE = None

display(transcript_candidate_summary.head(30))
print(f"Transcript candidate files: {len(transcript_candidates):,}")
print(f"Transcript candidate directories: {len(directory_candidates):,}")
print(f"Selected transcript source: {TRANSCRIPT_SOURCE}")

,parent,file_count,total_size_mb,extensions
0,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts,22821,573.0229,.csv


Transcript candidate files: 22,821
Transcript candidate directories: 1
Selected transcript source: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_transcripts


# Step 7.3 — Manual Transcript Source Selection

When multiple transcript candidates are displayed, select the correct raw transcript file or folder manually.

Update `TRANSCRIPT_SOURCE_OVERRIDE` in Step 3.1 using the exact Windows path.

Example:

    TRANSCRIPT_SOURCE_OVERRIDE = Path(r"C:\Users\USER\...\train_transcripts")

After updating the path, rerun the notebook from Step 3.1.

The True Turn Parser must not begin until the transcript source is confirmed.

# Step 8 — Training Schema Audit

## Step 8.1 — Schema Goals

This step will verify:

- Training feature and label row counts.
- Response identifier uniqueness.
- Session identifier availability.
- Learning-objective availability.
- Binary target validity.
- One-to-one feature-label merging.
- Absence of feature-only or label-only responses.
- Whether transcript content is embedded directly inside the feature table.

In [10]:
train_features = pd.read_csv(TRAIN_FEATURES_PATH, low_memory=False)
train_labels = pd.read_csv(TRAIN_LABELS_PATH, low_memory=False)

feature_response_col = pick_column(train_features, ["response_id", "responseid"], "train_features")
feature_session_col = pick_column(train_features, ["session_id", "sessionid"], "train_features")
feature_objective_col = pick_column(train_features, ["learning_objective", "objective", "objective_text"], "train_features")
embedded_transcript_col = pick_column(train_features, ["transcript", "utterances", "dialogue", "conversation", "messages"], "train_features", required=False)

label_response_col = pick_column(train_labels, ["response_id", "responseid"], "train_labels")
label_target_col = pick_column(train_labels, ["is_correct", "correct", "target", "label"], "train_labels")

detected_columns = {
    "feature_response_column": feature_response_col,
    "feature_session_column": feature_session_col,
    "feature_objective_column": feature_objective_col,
    "embedded_transcript_column": embedded_transcript_col,
    "label_response_column": label_response_col,
    "label_target_column": label_target_col
}

print(f"Train feature shape: {train_features.shape}")
print(f"Train label shape: {train_labels.shape}")
display(pd.DataFrame([detected_columns]))
print(f"Feature columns: {train_features.columns.tolist()}")
print(f"Label columns: {train_labels.columns.tolist()}")

Train feature shape: (35072, 4)
Train label shape: (35072, 2)


,feature_response_column,feature_session_column,feature_objective_column,embedded_transcript_column,label_response_column,label_target_column
0,response_id,session_id,learning_objective,None,response_id,is_correct


Feature columns: ['response_id', 'session_id', 'learning_objective_id', 'learning_objective']
Label columns: ['response_id', 'is_correct']


In [11]:
features_core = train_features[[feature_response_col, feature_session_col, feature_objective_col]].copy()
features_core.columns = ["response_id", "session_id", "learning_objective"]

labels_core = train_labels[[label_response_col, label_target_col]].copy()
labels_core.columns = ["response_id", "target"]

assert features_core["response_id"].notna().all(), "Missing response_id found in train features"
assert features_core["session_id"].notna().all(), "Missing session_id found in train features"
assert features_core["learning_objective"].notna().all(), "Missing learning objective found in train features"
assert labels_core["response_id"].notna().all(), "Missing response_id found in train labels"
assert labels_core["target"].notna().all(), "Missing target found in train labels"

features_core["response_id"] = features_core["response_id"].astype(str)
features_core["session_id"] = features_core["session_id"].astype(str)
features_core["learning_objective"] = features_core["learning_objective"].astype(str).str.strip()
labels_core["response_id"] = labels_core["response_id"].astype(str)
labels_core["target"] = pd.to_numeric(labels_core["target"], errors="raise").astype(int)

assert features_core["response_id"].is_unique, "Duplicate response_id found in train features"
assert labels_core["response_id"].is_unique, "Duplicate response_id found in train labels"
assert features_core["learning_objective"].ne("").all(), "Empty learning objective found"
assert set(labels_core["target"].unique().tolist()).issubset({0, 1}), f"Unexpected target values found: {sorted(labels_core['target'].unique().tolist())}"

feature_label_merge = features_core.merge(labels_core, on="response_id", how="outer", validate="one_to_one", indicator=True)
merge_status_counts = feature_label_merge["_merge"].value_counts(dropna=False).to_dict()

assert feature_label_merge["_merge"].eq("both").all(), f"Feature-label merge mismatch found: {merge_status_counts}"

canonical_response_table = feature_label_merge.drop(columns="_merge").copy()
canonical_response_table["target"] = canonical_response_table["target"].astype(int)

schema_audit = {
    "train_feature_rows": len(train_features),
    "train_label_rows": len(train_labels),
    "merged_response_rows": len(canonical_response_table),
    "unique_responses": canonical_response_table["response_id"].nunique(),
    "unique_sessions": canonical_response_table["session_id"].nunique(),
    "unique_objectives": canonical_response_table["learning_objective"].nunique(),
    "positive_rows": int(canonical_response_table["target"].sum()),
    "negative_rows": int((1 - canonical_response_table["target"]).sum()),
    "positive_rate": float(canonical_response_table["target"].mean()),
    "feature_label_merge_status": merge_status_counts,
    "detected_columns": detected_columns
}

write_json(schema_audit, SETUP_OUTPUT_DIR / "schema_audit.json")
display(pd.DataFrame([schema_audit]).T.rename(columns={0: "value"}))

,value
train_feature_rows,35072
train_label_rows,35072
merged_response_rows,35072
unique_responses,35072
unique_sessions,22821
unique_objectives,398
positive_rows,24637
negative_rows,10435
positive_rate,0.702469
feature_label_merge_status,"{'both': 35072, 'left_only': 0, 'right_only': 0}"


# Step 9 — Frozen Five-Fold Recovery

## Step 9.1 — Why the Previous Folds Are Retained

The Scratch Mastery pipeline will rebuild the data and modelling workflow from the beginning.

However, the grouped validation folds will remain unchanged so that every new result can be compared fairly with the previous baseline.

## Step 9.2 — Recovery Strategy

The notebook will attempt the following:

1. Search the previous fold-design directory for a complete fold manifest.
2. Validate each candidate against the current training response IDs.
3. If no complete manifest is found, recover fold membership from per-fold validation-ID files.
4. Expand session-level fold assignments to response-level assignments when necessary.
5. confirm that every response belongs to exactly one fold.
6. Confirm that every session belongs to exactly one fold.

In [12]:
RESPONSE_ID_CANDIDATES = ["response_id", "responseid", "validation_id", "id"]
SESSION_ID_CANDIDATES = ["session_id", "sessionid"]
FOLD_COLUMN_CANDIDATES = ["fold", "fold_id", "cv_fold", "validation_fold", "split"]
TABLE_EXTENSIONS = {".csv", ".parquet", ".pq", ".json", ".jsonl"}

def normalise_fold_table(table, table_name, inferred_fold=None):
    prepared = prepare_table(table)
    response_column = pick_column(prepared, RESPONSE_ID_CANDIDATES, table_name, required=False)
    session_column = pick_column(prepared, SESSION_ID_CANDIDATES, table_name, required=False)
    fold_column = pick_column(prepared, FOLD_COLUMN_CANDIDATES, table_name, required=False)
    if response_column is None and session_column is None:
        return None
    if fold_column is None and inferred_fold is None:
        return None
    if response_column is not None:
        normalised = prepared[[response_column]].copy()
        normalised.columns = ["response_id"]
        normalised["response_id"] = normalised["response_id"].astype(str)
        normalised = normalised.merge(features_core[["response_id", "session_id"]], on="response_id", how="left", validate="many_to_one")
    else:
        normalised = prepared[[session_column]].copy()
        normalised.columns = ["session_id"]
        normalised["session_id"] = normalised["session_id"].astype(str)
        normalised = features_core[["response_id", "session_id"]].merge(normalised.drop_duplicates(), on="session_id", how="inner", validate="many_to_one")
    normalised["fold"] = prepared[fold_column].values if fold_column is not None and len(prepared) == len(normalised) else inferred_fold
    if fold_column is not None and response_column is None:
        session_fold_map = prepared[[session_column, fold_column]].copy()
        session_fold_map.columns = ["session_id", "fold"]
        session_fold_map["session_id"] = session_fold_map["session_id"].astype(str)
        normalised = features_core[["response_id", "session_id"]].merge(session_fold_map.drop_duplicates(), on="session_id", how="inner", validate="many_to_one")
    normalised["fold"] = pd.to_numeric(normalised["fold"], errors="raise").astype(int)
    return normalised[["response_id", "session_id", "fold"]].drop_duplicates().reset_index(drop=True)

def validate_complete_fold_manifest(candidate, expected_response_ids):
    if candidate is None or candidate.empty:
        return False
    if candidate["response_id"].duplicated().any():
        return False
    if set(candidate["response_id"]) != expected_response_ids:
        return False
    if set(candidate["fold"].unique().tolist()) != {0, 1, 2, 3, 4}:
        return False
    if candidate.groupby("session_id")["fold"].nunique().max() != 1:
        return False
    return True

expected_response_ids = set(features_core["response_id"].tolist())
ready_fold_candidates = []

if BASELINE_FOLD_DESIGN_ROOT.exists():
    for candidate_path in BASELINE_FOLD_DESIGN_ROOT.rglob("*"):
        if not candidate_path.is_file() or candidate_path.suffix.lower() not in TABLE_EXTENSIONS:
            continue
        try:
            candidate_table = read_table(candidate_path)
            normalised_candidate = normalise_fold_table(candidate_table, str(candidate_path))
            if validate_complete_fold_manifest(normalised_candidate, expected_response_ids):
                ready_fold_candidates.append({"path": candidate_path, "table": normalised_candidate})
        except Exception:
            continue

print(f"Valid complete fold manifests found: {len(ready_fold_candidates)}")

if ready_fold_candidates:
    display(pd.DataFrame([{"path": str(item["path"]), "rows": len(item["table"])} for item in ready_fold_candidates]))

Valid complete fold manifests found: 1


,path,rows
0,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis\02_oof_analysis\00_fold_design\fo...,35072


In [13]:
if ready_fold_candidates:
    selected_fold_candidate = ready_fold_candidates[0]
    frozen_folds = selected_fold_candidate["table"].copy()
    fold_source_description = f"Complete fold manifest: {selected_fold_candidate['path']}"
else:
    validation_file_candidates = []
    search_root = BASELINE_FOLD_MODELS_ROOT if BASELINE_FOLD_MODELS_ROOT.exists() else BASELINE_REFERENCE_ROOT
    for candidate_path in search_root.rglob("*"):
        if not candidate_path.is_file() or candidate_path.suffix.lower() not in TABLE_EXTENSIONS:
            continue
        path_text = str(candidate_path).lower()
        fold_match = re.search(r"fold[_\-\s]?(\d+)", path_text)
        if fold_match is None or "validation" not in path_text:
            continue
        try:
            fold_number = int(fold_match.group(1))
            candidate_table = read_table(candidate_path)
            normalised_candidate = normalise_fold_table(candidate_table, str(candidate_path), inferred_fold=fold_number)
            if normalised_candidate is not None and not normalised_candidate.empty:
                validation_file_candidates.append({"fold": fold_number, "path": candidate_path, "table": normalised_candidate})
        except Exception:
            continue

    validation_candidate_summary = pd.DataFrame([{"fold": item["fold"], "path": str(item["path"]), "rows": len(item["table"])} for item in validation_file_candidates])
    display(validation_candidate_summary.sort_values(["fold", "rows"], ascending=[True, False]) if not validation_candidate_summary.empty else validation_candidate_summary)

    selected_fold_parts = []

    for fold_number in range(5):
        fold_candidates = [item for item in validation_file_candidates if item["fold"] == fold_number]
        assert fold_candidates, f"No validation-ID file was found for fold {fold_number}"
        selected_candidate = sorted(fold_candidates, key=lambda item: len(item["table"]), reverse=True)[0]
        selected_fold_parts.append(selected_candidate["table"])

    frozen_folds = pd.concat(selected_fold_parts, ignore_index=True)
    fold_source_description = "Reconstructed from per-fold validation-ID files"

frozen_folds = frozen_folds[["response_id", "session_id", "fold"]].drop_duplicates().reset_index(drop=True)

print(f"Fold source: {fold_source_description}")
print(f"Recovered fold rows: {len(frozen_folds):,}")
display(frozen_folds.head())

Fold source: Complete fold manifest: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis\02_oof_analysis\00_fold_design\fold_manifest.parquet
Recovered fold rows: 35,072


,response_id,session_id,fold
0,aaaavsh,bcaufvc,0
1,aaabhzi,eyutanf,4
2,aaahpnz,juptkxd,2
3,aaajpom,ntwkcfj,0
4,aaamwux,jqriibm,0


In [14]:
assert frozen_folds["response_id"].notna().all(), "Missing response_id found in frozen fold manifest"
assert frozen_folds["session_id"].notna().all(), "Missing session_id found in frozen fold manifest"
assert frozen_folds["fold"].notna().all(), "Missing fold assignment found"
assert frozen_folds["response_id"].is_unique, "A response appears more than once in the frozen folds"
assert set(frozen_folds["fold"].unique().tolist()) == {0, 1, 2, 3, 4}, f"Unexpected fold values: {sorted(frozen_folds['fold'].unique().tolist())}"

fold_response_ids = set(frozen_folds["response_id"].tolist())
missing_from_folds = expected_response_ids - fold_response_ids
unexpected_in_folds = fold_response_ids - expected_response_ids

assert not missing_from_folds, f"{len(missing_from_folds)} training responses are missing from the frozen folds"
assert not unexpected_in_folds, f"{len(unexpected_in_folds)} unknown responses were found in the frozen folds"

session_fold_counts = frozen_folds.groupby("session_id")["fold"].nunique()
assert session_fold_counts.max() == 1, "At least one session appears in multiple folds"

fold_evaluation_table = frozen_folds.merge(labels_core, on="response_id", how="left", validate="one_to_one")
fold_summary = fold_evaluation_table.groupby("fold").agg(responses=("response_id", "size"), sessions=("session_id", "nunique"), positive_rows=("target", "sum"), positive_rate=("target", "mean")).reset_index()
fold_summary["negative_rows"] = fold_summary["responses"] - fold_summary["positive_rows"]

FROZEN_FOLD_MANIFEST_PATH = save_dataframe(frozen_folds, SETUP_OUTPUT_DIR / "frozen_fold_manifest.parquet")
fold_summary.to_csv(SETUP_OUTPUT_DIR / "frozen_fold_summary.csv", index=False)

display(fold_summary)
print(f"Frozen fold manifest saved: {FROZEN_FOLD_MANIFEST_PATH}")
print(f"Maximum folds assigned to one session: {session_fold_counts.max()}")

,fold,responses,sessions,positive_rows,positive_rate,negative_rows
0,0,6980,4590,4890,0.700573,2090
1,1,7018,4569,4905,0.698917,2113
2,2,7019,4570,4989,0.710785,2030
3,3,7011,4541,4905,0.699615,2106
4,4,7044,4551,4948,0.702442,2096


Frozen fold manifest saved: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\frozen_fold_manifest.parquet
Maximum folds assigned to one session: 1


# Step 10 — Environment Manifest

## Step 10.1 — Environment Information

This step records:

- Python version.
- Python executable.
- Operating system.
- CPU information.
- Installed package versions.
- Random seed.
- Current working directory.

Missing modelling packages will be reported but will not be installed automatically.

In [15]:
PACKAGE_NAMES = ["numpy", "pandas", "pyarrow", "scikit-learn", "scipy", "matplotlib", "jupyterlab", "notebook", "ipykernel", "torch", "transformers", "sentence-transformers", "datasets", "accelerate", "lightgbm", "xgboost"]

package_versions = {}

for package_name in PACKAGE_NAMES:
    try:
        package_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        package_versions[package_name] = None

environment_manifest = {
    "created_at": datetime.now().astimezone().isoformat(),
    "python_version": sys.version,
    "python_major_minor": f"{sys.version_info.major}.{sys.version_info.minor}",
    "python_executable": sys.executable,
    "platform": platform.platform(),
    "operating_system": platform.system(),
    "operating_system_release": platform.release(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "project_root": str(PROJECT_ROOT),
    "current_working_directory": str(Path.cwd()),
    "random_seed": SEED,
    "package_versions": package_versions
}

write_json(environment_manifest, SETUP_OUTPUT_DIR / "environment_manifest.json")

package_table = pd.DataFrame([{"package": package_name, "installed": package_version is not None, "version": package_version} for package_name, package_version in package_versions.items()])

display(package_table)
print(f"Python version: {environment_manifest['python_major_minor']}")

,package,installed,version
0,numpy,True,2.2.6
1,pandas,True,2.3.3
2,pyarrow,True,25.0.0
3,scikit-learn,True,1.7.2
4,scipy,True,1.15.3
5,matplotlib,True,3.10.9
6,jupyterlab,True,4.5.10
7,notebook,True,7.5.5
8,ipykernel,True,7.3.0
9,torch,True,2.11.0+cu128


Python version: 3.10


# Step 11 — Hardware Manifest

## Step 11.1 — Hardware Information

This step records:

- CPU count.
- Total and available system memory.
- Total and available project-disk storage.
- PyTorch availability.
- CUDA availability.
- CUDA version.
- GPU name.
- GPU VRAM.
- GPU compute capability.

CUDA is not required for data-foundation or EDA work. It will be required later for transformer training.

In [16]:
hardware_manifest = {
    "created_at": datetime.now().astimezone().isoformat(),
    "cpu_count": os.cpu_count(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "project_disk_total_gb": round(shutil.disk_usage(PROJECT_ROOT).total / (1024 ** 3), 2),
    "project_disk_free_gb": round(shutil.disk_usage(PROJECT_ROOT).free / (1024 ** 3), 2),
    "torch_installed": False,
    "cuda_available": False,
    "cuda_device_count": 0,
    "gpu_devices": []
}

try:
    import psutil
    hardware_manifest["system_memory_total_gb"] = round(psutil.virtual_memory().total / (1024 ** 3), 2)
    hardware_manifest["system_memory_available_gb"] = round(psutil.virtual_memory().available / (1024 ** 3), 2)
except ImportError:
    hardware_manifest["system_memory_total_gb"] = None
    hardware_manifest["system_memory_available_gb"] = None

try:
    import torch
    hardware_manifest["torch_installed"] = True
    hardware_manifest["torch_version"] = torch.__version__
    hardware_manifest["torch_cuda_version"] = torch.version.cuda
    hardware_manifest["cuda_available"] = torch.cuda.is_available()
    hardware_manifest["cuda_device_count"] = torch.cuda.device_count()
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            device_properties = torch.cuda.get_device_properties(device_index)
            hardware_manifest["gpu_devices"].append({"device_index": device_index, "name": device_properties.name, "total_vram_gb": round(device_properties.total_memory / (1024 ** 3), 2), "compute_capability": f"{device_properties.major}.{device_properties.minor}"})
except ImportError:
    pass

write_json(hardware_manifest, SETUP_OUTPUT_DIR / "hardware_manifest.json")

hardware_display = {
    "CPU count": hardware_manifest["cpu_count"],
    "System memory total GB": hardware_manifest.get("system_memory_total_gb"),
    "System memory available GB": hardware_manifest.get("system_memory_available_gb"),
    "Project disk free GB": hardware_manifest["project_disk_free_gb"],
    "PyTorch installed": hardware_manifest["torch_installed"],
    "CUDA available": hardware_manifest["cuda_available"],
    "CUDA device count": hardware_manifest["cuda_device_count"],
    "GPU devices": hardware_manifest["gpu_devices"]
}

display(pd.DataFrame([hardware_display]).T.rename(columns={0: "value"}))

,value
CPU count,16
System memory total GB,31.58
System memory available GB,16.31
Project disk free GB,12.79
PyTorch installed,True
CUDA available,True
CUDA device count,1
GPU devices,"[{'device_index': 0, 'name': 'NVIDIA GeForce RTX 5060 Ti', 'total_vram_gb': 7.96, 'compute_capability': '12.0'}]"


# Step 12 — Git Safety Audit

## Step 12.1 — Safety Checks

This step does not modify Git.

It only checks:

- Current Git branch.
- Current Git commit.
- Working-tree status.
- Whether the raw dataset root is ignored.
- Whether the Scratch Mastery output root is ignored.

Generated data and model artifacts must not be committed accidentally.

In [17]:
git_branch_result = run_command(["git", "branch", "--show-current"], cwd=PROJECT_ROOT)
git_commit_result = run_command(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT)
git_status_result = run_command(["git", "status", "--short"], cwd=PROJECT_ROOT)

scratch_relative_path = SCRATCH_OUTPUT_ROOT.relative_to(PROJECT_ROOT)
dataset_relative_path = DATA_ROOT.relative_to(PROJECT_ROOT)

scratch_ignore_result = run_command(["git", "check-ignore", "-q", str(scratch_relative_path)], cwd=PROJECT_ROOT)
dataset_ignore_result = run_command(["git", "check-ignore", "-q", str(dataset_relative_path)], cwd=PROJECT_ROOT)

git_manifest = {
    "created_at": datetime.now().astimezone().isoformat(),
    "branch": git_branch_result["stdout"],
    "commit": git_commit_result["stdout"],
    "working_tree_status": git_status_result["stdout"],
    "working_tree_clean": git_status_result["stdout"] == "",
    "scratch_output_ignored": scratch_ignore_result["returncode"] == 0,
    "dataset_root_ignored": dataset_ignore_result["returncode"] == 0
}

write_json(git_manifest, SETUP_OUTPUT_DIR / "git_manifest.json")
display(pd.DataFrame([git_manifest]).T.rename(columns={0: "value"}))

if not git_manifest["scratch_output_ignored"]:
    print("WARNING: scratch_mastery_outputs is not ignored by Git.")

if not git_manifest["dataset_root_ignored"]:
    print("WARNING: Trace-The-Race-Dataset is not ignored by Git.")

,value
created_at,2026-08-07T02:15:21.966982+06:00
branch,main
commit,f4ce02bca1682c5d29b50e34153f38c8658584dd
working_tree_status,M notebook_advance/00_project_setup/00_environment_and_paths.ipynb\n?? PDF/Trace_the_Ace_Baseline_Failure_Analysis_10_Pages.pdf\n?? PDF/Trace_the_...
working_tree_clean,False
scratch_output_ignored,False
dataset_root_ignored,False


# Step 13 — Canonical Path Registry

## Step 13.1 — Registry Purpose

A reusable path registry will be saved for later notebooks.

The registry will include:

- Official raw-data paths.
- Baseline reference paths.
- Scratch output paths.
- Frozen-fold manifest path.
- Transcript source path or embedded transcript column.
- Baseline notebook and report paths.

In [18]:
if TRANSCRIPT_SOURCE is not None:
    TRANSCRIPT_SOURCE_TYPE = "external_path"
    TRANSCRIPT_SOURCE_VALUE = str(TRANSCRIPT_SOURCE)
elif embedded_transcript_col is not None:
    TRANSCRIPT_SOURCE_TYPE = "embedded_feature_column"
    TRANSCRIPT_SOURCE_VALUE = str(TRAIN_FEATURES_PATH)
else:
    TRANSCRIPT_SOURCE_TYPE = "not_selected"
    TRANSCRIPT_SOURCE_VALUE = None

path_registry = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "old_notebook_root": str(OLD_NOTEBOOK_ROOT),
    "model_analysis_root": str(MODEL_ANALYSIS_ROOT),
    "baseline_reference_root": str(BASELINE_REFERENCE_ROOT),
    "advanced_notebook_root": str(ADVANCED_NOTEBOOK_ROOT),
    "scratch_output_root": str(SCRATCH_OUTPUT_ROOT),
    "setup_output_dir": str(SETUP_OUTPUT_DIR),
    "train_features_path": str(TRAIN_FEATURES_PATH),
    "train_labels_path": str(TRAIN_LABELS_PATH),
    "submission_format_path_1": str(SUBMISSION_FORMAT_PATH_1),
    "submission_format_path_2": str(SUBMISSION_FORMAT_PATH_2),
    "old_master_dataset_path": str(OLD_MASTER_DATASET_PATH),
    "baseline_fold_design_root": str(BASELINE_FOLD_DESIGN_ROOT),
    "baseline_fold_models_root": str(BASELINE_FOLD_MODELS_ROOT),
    "baseline_oof_root": str(BASELINE_OOF_ROOT),
    "frozen_fold_manifest_path": str(FROZEN_FOLD_MANIFEST_PATH),
    "transcript_source_type": TRANSCRIPT_SOURCE_TYPE,
    "transcript_source": TRANSCRIPT_SOURCE_VALUE,
    "embedded_transcript_column": embedded_transcript_col,
    "baseline_model_notebook": str(BASELINE_MODEL_NOTEBOOK),
    "baseline_analysis_notebook": str(BASELINE_ANALYSIS_NOTEBOOK),
    "baseline_diagnostic_notebook": str(BASELINE_DIAGNOSTIC_NOTEBOOK),
    "baseline_report_root": str(BASELINE_REPORT_ROOT)
}

write_json(path_registry, SETUP_OUTPUT_DIR / "path_registry.json")
display(pd.DataFrame([{"name": key, "value": value} for key, value in path_registry.items()]))

,name,value
0,project_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition
1,data_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset
2,old_notebook_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook
3,model_analysis_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis
4,baseline_reference_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook\model_analysis\baselineanalysis
5,advanced_notebook_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\notebook_advance
6,scratch_output_root,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs
7,setup_output_dir,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup
8,train_features_path,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_features_TMQTWsB.csv
9,train_labels_path,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\Trace-The-Race-Dataset\train_labels_44ujmj2.csv


# Step 14 — Final Setup Scorecard

## Step 14.1 — Readiness Conditions

The True Turn Parser can begin only when all critical checks pass:

- Required project paths are valid.
- Feature-label schema validation passes.
- Frozen grouped folds are complete.
- No session appears in multiple folds.
- The transcript source is selected.
- Scratch outputs are ignored by Git.

Python 3.12 and CUDA are important for later modelling stages, but they are not required for the first data-foundation notebook.

In [19]:
DATA_PATHS_READY = missing_required_sources.empty
FEATURE_LABEL_SCHEMA_READY = feature_label_merge["_merge"].eq("both").all()
FROZEN_FOLDS_READY = frozen_folds["response_id"].is_unique and session_fold_counts.max() == 1 and set(frozen_folds["fold"].unique().tolist()) == {0, 1, 2, 3, 4}
TRANSCRIPT_SOURCE_READY = (TRANSCRIPT_SOURCE is not None and TRANSCRIPT_SOURCE.exists()) or embedded_transcript_col is not None
SCRATCH_OUTPUT_GIT_SAFE = git_manifest["scratch_output_ignored"]
DATASET_GIT_SAFE = git_manifest["dataset_root_ignored"]
PYTHON_312_READY = sys.version_info.major == 3 and sys.version_info.minor == 12
CUDA_READY = hardware_manifest["cuda_available"]
TURN_PARSER_READY_TO_START = DATA_PATHS_READY and FEATURE_LABEL_SCHEMA_READY and FROZEN_FOLDS_READY and TRANSCRIPT_SOURCE_READY and SCRATCH_OUTPUT_GIT_SAFE

setup_scorecard = pd.DataFrame([
    {"check": "Project and required paths", "status": "PASS" if DATA_PATHS_READY else "FAIL", "required_for_turn_parser": True},
    {"check": "Feature-label schema and merge", "status": "PASS" if FEATURE_LABEL_SCHEMA_READY else "FAIL", "required_for_turn_parser": True},
    {"check": "Frozen five-fold manifest", "status": "PASS" if FROZEN_FOLDS_READY else "FAIL", "required_for_turn_parser": True},
    {"check": "Transcript source selected", "status": "PASS" if TRANSCRIPT_SOURCE_READY else "PENDING", "required_for_turn_parser": True},
    {"check": "Scratch output ignored by Git", "status": "PASS" if SCRATCH_OUTPUT_GIT_SAFE else "FAIL", "required_for_turn_parser": True},
    {"check": "Dataset root ignored by Git", "status": "PASS" if DATASET_GIT_SAFE else "WARNING", "required_for_turn_parser": False},
    {"check": "Python 3.12 local environment", "status": "PASS" if PYTHON_312_READY else "WARNING", "required_for_turn_parser": False},
    {"check": "CUDA available", "status": "PASS" if CUDA_READY else "WARNING", "required_for_turn_parser": False}
])

setup_summary = {
    "created_at": datetime.now().astimezone().isoformat(),
    "data_paths_ready": DATA_PATHS_READY,
    "feature_label_schema_ready": FEATURE_LABEL_SCHEMA_READY,
    "frozen_folds_ready": FROZEN_FOLDS_READY,
    "transcript_source_ready": TRANSCRIPT_SOURCE_READY,
    "scratch_output_git_safe": SCRATCH_OUTPUT_GIT_SAFE,
    "dataset_git_safe": DATASET_GIT_SAFE,
    "python_312_ready": PYTHON_312_READY,
    "cuda_ready": CUDA_READY,
    "turn_parser_ready_to_start": TURN_PARSER_READY_TO_START,
    "response_rows": len(canonical_response_table),
    "session_count": canonical_response_table["session_id"].nunique(),
    "objective_count": canonical_response_table["learning_objective"].nunique(),
    "positive_rate": canonical_response_table["target"].mean(),
    "frozen_fold_manifest_path": str(FROZEN_FOLD_MANIFEST_PATH),
    "transcript_source_type": TRANSCRIPT_SOURCE_TYPE,
    "transcript_source": TRANSCRIPT_SOURCE_VALUE,
    "embedded_transcript_column": embedded_transcript_col
}

write_json(setup_summary, SETUP_OUTPUT_DIR / "setup_summary.json")

display(setup_scorecard)
print(f"TURN_PARSER_READY_TO_START = {TURN_PARSER_READY_TO_START}")

,check,status,required_for_turn_parser
0,Project and required paths,PASS,True
1,Feature-label schema and merge,PASS,True
2,Frozen five-fold manifest,PASS,True
3,Transcript source selected,PASS,True
4,Scratch output ignored by Git,FAIL,True
5,Dataset root ignored by Git,WARNING,False
6,Python 3.12 local environment,WARNING,False
7,CUDA available,PASS,False


TURN_PARSER_READY_TO_START = False


# Step 15 — Generated Setup Artifacts

## Step 15.1 — Artifact Verification

The final setup directory should contain the reusable manifests and audit outputs created by this notebook.

In [20]:
generated_setup_files = []

for generated_path in sorted(SETUP_OUTPUT_DIR.glob("*")):
    if generated_path.is_file():
        generated_setup_files.append({"file": generated_path.name, "size_kb": round(generated_path.stat().st_size / 1024, 2), "path": str(generated_path)})

generated_setup_table = pd.DataFrame(generated_setup_files)

display(generated_setup_table)
print(f"Setup output directory: {SETUP_OUTPUT_DIR}")
print(f"Generated setup artifacts: {len(generated_setup_table)}")

,file,size_kb,path
0,environment_manifest.json,1.23,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\environment_manifest.json
1,frozen_fold_manifest.parquet,600.60,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\frozen_fold_manifest.parquet
2,frozen_fold_summary.csv,0.27,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\frozen_fold_summary.csv
3,git_manifest.json,0.46,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\git_manifest.json
4,hardware_manifest.json,0.63,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\hardware_manifest.json
5,path_registry.json,3.43,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\path_registry.json
6,raw_data_inventory.csv,6528.46,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\raw_data_inventory.csv
7,schema_audit.json,0.67,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\schema_audit.json
8,setup_summary.json,0.87,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\setup_summary.json
9,source_fingerprints.json,0.62,C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup\source_fingerprints.json


Setup output directory: C:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Trace-The-Race-Competition\scratch_mastery_outputs\00_project_setup
Generated setup artifacts: 10


# Step 16 — Setup Completion Decision

## Step 16.1 — Expected Setup Outputs

The completed setup should create the following artifacts inside `scratch_mastery_outputs/00_project_setup/`:

- `environment_manifest.json`
- `frozen_fold_manifest.parquet` or its CSV fallback
- `frozen_fold_summary.csv`
- `git_manifest.json`
- `hardware_manifest.json`
- `path_registry.json`
- `raw_data_inventory.csv`
- `schema_audit.json`
- `setup_summary.json`
- `source_fingerprints.json`

## Step 16.2 — Final Readiness Requirement

The required final flag is:

    TURN_PARSER_READY_TO_START = True

When the flag is `False`, review the final scorecard.

The most likely causes are:

1. The transcript source has not been selected.
2. `scratch_mastery_outputs/` is not ignored by Git.
3. The frozen-fold files could not be recovered.
4. A required source path is incorrect.
5. The feature and label tables do not merge one-to-one.

## Step 16.3 — Next Notebook

After all critical checks pass, the next notebook will be:

    notebook_advance/01_data_foundation/01_true_turn_parser.ipynb